# 02.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/06-polynomial-and-overfitting/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
# ShopWave ran 12 ad campaigns last year (spend in £k) and counted the extra orders each brought.
ads = pd.DataFrame({"ad_spend_k": [0.5, 1, 1.5, 2, 3, 4, 5, 6, 7, 8, 9, 10],
                    "extra_orders": [42, 62, 71, 78, 100, 112, 135, 158, 155, 165, 184, 193]})
# Thirty campaigns from this year, held back to test on (same process, new noise)
rng = np.random.default_rng(3)
test_x = np.sort(rng.uniform(0.5, 10, 30))
ads_test = pd.DataFrame({"ad_spend_k": test_x, "extra_orders": (60 * np.sqrt(test_x) + rng.normal(0, 8, 30)).round(0)})
ads


from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

def poly_model(degree):
    """Scale x, build x, x², …, x^degree, fit a line through them."""
    return make_pipeline(StandardScaler(), PolynomialFeatures(degree), LinearRegression())

from sklearn.metrics import mean_absolute_error
X_tr, y_tr = ads[["ad_spend_k"]], ads["extra_orders"]
X_te, y_te = ads_test[["ad_spend_k"]], ads_test["extra_orders"]

## Exercise 1

$x = 2$: $[1, 2, 4, 8]$. $x = 0.5$: $[1, 0.5, 0.25, 0.125]$. Note how values below 1 *shrink* with each power and values above 1 explode; that spread is why we scale first.

In [2]:
print(PolynomialFeatures(3).fit_transform(pd.DataFrame({"ad_spend_k": [2.0, 0.5]})))

[[1.    2.    4.    8.   ]
 [1.    0.5   0.25  0.125]]


## Exercise 2

In [3]:
raw2 = make_pipeline(PolynomialFeatures(2), LinearRegression()).fit(X_tr, y_tr)
b0 = raw2[-1].intercept_; _, b1, b2 = raw2[-1].coef_
print(f"extra_orders ≈ {b0:.1f} + {b1:.2f}·x + ({b2:.2f})·x²")

extra_orders ≈ 35.1 + 23.67·x + (-0.81)·x²


$b_2$ is **negative**: each extra £1k of ad spend adds a little less than the last. A negative squared term is exactly "diminishing returns", and the line could not express it.

## Exercise 3

In [4]:
rng = np.random.default_rng(0)
xb = rng.uniform(0.5, 10, 120)
big = pd.DataFrame({"ad_spend_k": xb, "extra_orders": 60 * np.sqrt(xb) + rng.normal(0, 8, 120)})
for name, X, y in [("12 rows", X_tr, y_tr), ("120 rows", big[["ad_spend_k"]], big["extra_orders"])]:
    m = poly_model(9).fit(X, y)
    print(f"degree 9 trained on {name}: test MAE {mean_absolute_error(y_te, m.predict(X_te)):.1f}")

degree 9 trained on 12 rows: test MAE 7.9
degree 9 trained on 120 rows: test MAE 5.9


With 120 rows the degree-9 curve can no longer thread through every point; the noise averages out and the test error drops to the degree-2 level. Overfitting is a ratio: model freedom versus data. More data shrinks it.

## Exercise 4

In [5]:
for d in [1, 2, 3, 5, 7, 9]:
    m = poly_model(d).fit(X_tr, y_tr)
    print(f"degree {d}: MAE on the training rows {mean_absolute_error(y_tr, m.predict(X_tr)):.2f}")

degree 1: MAE on the training rows 5.90
degree 2: MAE on the training rows 3.99
degree 3: MAE on the training rows 3.62
degree 5: MAE on the training rows 3.56
degree 7: MAE on the training rows 2.21
degree 9: MAE on the training rows 0.82


The highest degree "wins" with an error near zero. It is worthless because you asked the model about rows it has already seen; a lookup table would score the same. A score only means something on data the model did not train on.